# Tiny GPT Decoder 11 (Generation Decoding - Beam Search)

In the 11th version we are focusing on inference using beam search. Beam search will keep the 2 best word combination.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

The followings code is to run to activate the GPU

In [2]:
# 1. Check for NVIDIA CUDA (Linux / Colab)
if torch.cuda.is_available():
    device = torch.device("cuda")
# 2. Check for Apple Silicon MPS (M2 Mac)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
# 3. Fallback to CPU
else:
    device = torch.device("cpu")

In [3]:
print(f'Using device: {device}')

Using device: mps


## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [4]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [5]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing and Tokenization

In [ ]:
# Tokenization

def tiny_tokenization(corpus_or_prompt):
  return corpus_or_prompt.split()

# Split corpus into words
word_tokens = tiny_tokenization(corpus)

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [ ]:
vocab_size

In [ ]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


## Class and Function Setup

In [ ]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  return X, Y

In [ ]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [ ]:
class TinyGPT(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        # Initialize trainable components here
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.WQ = nn.Parameter(torch.randn(d_model, d_model))
        self.WK = nn.Parameter(torch.randn(d_model, d_model))
        self.WV = nn.Parameter(torch.randn(d_model, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False),
            nn.ReLU(),
            nn.Linear(d_model, d_model, bias=False)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, token_ids):

        # Perform Transformer calculations here
        # Word embeddings + positional encoding
        X_input = self.embeddings(token_ids)
        position = torch.arange(
            len(token_ids),
            dtype=X_input.dtype,
            device=X_input.device
        ).reshape(-1, 1)
        X = X_input + position

        # Q, K, V projections
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        # Scaled dot-product attention
        attention_score = Q @ K.T
        stable_factor = self.WQ.shape[-1] ** 0.5
        attention_stable = attention_score / stable_factor

        # Causal masking + softmax
        causal_mask = torch.triu(
            torch.full_like(attention_stable, float("-inf")),
            diagonal=1
        )
        attention_stable_mask = attention_stable + causal_mask
        attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

        # Attention output + residual + norm1
        attention_output = attention_softmax @ V

        # Residual Layer
        attention_output_residual = attention_output + X

        # norm1 layer
        attention_output_residual_scaled = self.norm1(attention_output_residual)

        # FFN + residual + norm2
        ffn_1 = self.ffn(attention_output_residual_scaled)

        # Residual 2 Layer
        seond_residual = ffn_1 + attention_output_residual_scaled

        # Norm 2 Layer
        transformer_output = self.norm2(seond_residual)

        # Vocabulary projection
        logits = transformer_output @ self.embeddings.weight.T

        return logits

### Training Setup

In [ ]:
def llm_train(corpus_in_id, block_size, model, criterion, optimizer, epoch=10):

  # Get N
  N = get_N(corpus_in_id, block_size=block_size)

  for i in range(epoch):

    print(f"epoch {i}")

    for j in range(N):

      # Get block
      X_id, Y_id = get_sequence_block(
          block_size=block_size,
          start_position=j * block_size,
          corpus_in_id=corpus_in_id
      )
      X_tensor = torch.tensor(X_id, dtype=torch.long, device=device)
      Y_tensor = torch.tensor(Y_id, dtype=torch.long, device=device)

      optimizer.zero_grad()

      logits = model(X_tensor)
      loss = criterion(logits, Y_tensor)
      print(f"Block {j} : Loss = {loss.item():.4f}")

      loss.backward()
      optimizer.step()

### Train the Entire Corpus

In [ ]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
llm_train(
    corpus_in_id=corpus_in_id,
    block_size=block_size,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    epoch=5000
)

We intentionally increase the epoch to get 100% accuracy. This is for model evaluation and inference later.

In [ ]:
def evaluate_blocks(model, criterion, corpus_in_id, block_size, N):
    model.eval()

    with torch.no_grad():
        for i in range(N):
            start = i * block_size

            X_id, Y_id = get_sequence_block(
                block_size=block_size,
                start_position=start,
                corpus_in_id=corpus_in_id
            )

            X = torch.tensor(X_id, dtype=torch.long, device=device)
            Y = torch.tensor(Y_id, dtype=torch.long, device=device)

            logits = model(X)
            loss = criterion(logits, Y)

            print(f"Block {i}: Loss = {loss.item():.4f}")

            predicted_ids = torch.argmax(logits, dim=-1)
            correct = (predicted_ids == Y).sum().item()
            accuracy = correct / len(Y)

            print(f"Block {i}: Accuracy = {accuracy:.4f}")

            print("-" * 50)
            print(f"Block {i}")
            for k in range(5):
              print(f"Position {k}, expected = {id_to_words_ref[int(Y[k])]}, predicted = {id_to_words_ref[int(predicted_ids[k])]}")
            print("-" * 50)

In [ ]:
N = get_N(corpus_in_id, block_size=block_size)


In [ ]:
evaluate_blocks(model=model, criterion=criterion, corpus_in_id=corpus_in_id, block_size=block_size, N=N)

## Inference Decoding - Beam Search

During inference, there are many decoding method:

- Greedy
- Sampling
- Temperature Sampling
- Top K Sampling
- Top P Sampling
- Beam Search

In this exercise, we will be looking at Beam Search.

## Beam Search

Beam Search is a decoding strategy that keeps track of the **best candidate sequences** during token generation.

We first define the **beam width (B)**. For example, if `B = 2`, we retain the **2 sequences with the highest cumulative beam scores** at each generation step.

### Step 1: Generate the First Token

1. Pass the prompt into the model to obtain the logits for the next token.
2. Apply softmax to obtain the probability distribution over the entire vocabulary.
3. Select the 2 tokens with the highest probabilities (`B = 2`).
4. Calculate the beam score:

$$\text{Beam Score} = 1.0 \times \text{Token Probability}$$

5. Store the 2 sequences together with their beam scores.

The initial beam score is `1.0` because no tokens have been generated yet.

### Step 2: Generate Subsequent Tokens

1. Take each of the 2 stored sequences and pass it separately into the model.
2. Each sequence produces its own next-token probability distribution using softmax.
3. Consider all possible next tokens in the vocabulary for **each stored sequence**.
4. Calculate the new cumulative beam score:

$$\text{New Beam Score} = \text{Previous Beam Score} \times \text{Next-Token Probability}$$

5. Compare the new beam scores across **all candidate sequences**.
6. Retain only the 2 sequences with the highest cumulative beam scores, together with their updated scores.
7. For each beam, we have B number of best candidate and later whenw e done for all beam, we will seelct teh best B globally.

### Step 3: Repeat

Repeat Step 2 until the required number of tokens has been generated or a stopping condition is reached.

### Key Points

- **Beam width** determines how many candidate sequences are retained at each step.
- Each beam stores the **complete sequence and its cumulative score**.
- Each beam generates its own softmax probability distribution because its token sequence may be different.
- Selection is based on the **cumulative beam score**, not just the probability of the latest token.
- With beam width = 2, we always retain at most 2 active sequences, regardless of how many candidates are considered.
- Beam Search is deterministic for a fixed model and settings, unlike Top-K and Top-P sampling.

### Functions for Inference

In [ ]:
def prompt_processing(user_prompt_text):
  user_prompt_text = user_prompt_text.split()
  prompt_id = [words_to_id_ref[each] for each in user_prompt_text]
  return torch.tensor(prompt_id, dtype=torch.long, device=device)

### Inference 1 - Top P Sampling

In [ ]:
def beam_search_logits(X, model, tau=1):
  model.eval()

  # the code below prevent torch for tracking the gradient since we are inferencing and not training
  with torch.no_grad():
    prompt_logits = model(X)

  # Set the temperature for the last logits
  predict_logits_before_temperature = prompt_logits[-1]
  predict_logits = predict_logits_before_temperature / tau

  # get the last token prediction
  next_token_probs = torch.softmax(predict_logits, dim=-1)

  return next_token_probs

In [ ]:
def beam_search(user_prompt, max_token, model, B, id_to_words_dict, tau=1):
  X = prompt_processing(user_prompt)
  beam_store = []

  for i in range(max_token):

    # check if it is first generation as we need to set teh previous beam score
    if i == 0:
      previous_beam_score = 1

      # get the last token prediction
      next_token_probs = beam_search_logits(X, model, tau=tau)

      # predicting token after
      # token = [id_to_words_dict[each.item()] for each in X]
      # print(f"Predicting: last token of the prompt {token}")

      # Get the best B softmax
      selected_probs, selected_token_ID = torch.topk(next_token_probs, k=B)
      selected_beam_score = previous_beam_score * selected_probs
      #print(selected_probs, selected_token_ID)

      for j in range(B):
        beam_store.append((
            torch.cat((X, selected_token_ID[j].view(1))),
            selected_beam_score[j].item()
        ))

      # print beam_store
      print(beam_store)

    else:
      beam_temp = []
      for beam_X, beam_score in beam_store:
        next_token_probs = beam_search_logits(beam_X, model, tau=tau)

        # Get the best B softmax
        selected_probs, selected_token_ID = torch.topk(next_token_probs, k=B)
        selected_beam_score = beam_score * selected_probs

        for k in range(B):
          beam_temp.append((
              torch.cat((beam_X, selected_token_ID[k].view(1))),
              selected_beam_score[k].item()
          ))

      print("Beam Candidates")
      print("-----")
      print(beam_temp)
      print("=======")
      beam_store = sorted(beam_temp, key=lambda x: x[1], reverse=True)[:B]
      print("Best from temp")
      print("-----")
      print(beam_store)
      print("=======")

  best_beam = beam_store[0]
  X = best_beam[0]

  return X

In [ ]:
def generate_response(X, id_to_words_dict):
  response = [id_to_words_dict[each.item()] for each in X]
  print(f"Prompt: {user_prompt}")
  print(f"Response: {' '.join(response)}")

In [ ]:
user_prompt = "mary"
user_prompt

In [ ]:
max_token = 4
tau = 1
B = 2

In [ ]:
X = beam_search(user_prompt, max_token, model, B, id_to_words_ref, tau)

In [ ]:
generate_response(X, id_to_words_ref)

We still get a good prediction. Our softmax is too high for get variety. The following example we will increase temperature.

### Inference 2 - Tau = 10, B = 3

In [ ]:
user_prompt = "mary"
user_prompt

In [ ]:
max_token = 4
tau = 10
B = 3

In [ ]:
X = beam_search(user_prompt, max_token, model, B, id_to_words_ref, tau)

In [ ]:
generate_response(X, id_to_words_ref)

Even when you increase temperature, beam search still selects the highest-scoring sequences rather than randomly sampling.

Also, temperature changes the probability distribution but does not change the ranking of tokens within that distribution.

Therefore, your beam search may continue producing the same sequence across different temperatures.

This is expected behavior.


### Inference 3 -  Tau = 5, B = 3

In [ ]:
user_prompt = "mary had a"
user_prompt

In [ ]:
max_token = 2
tau = 5
B = 3

In [ ]:
X = beam_search(user_prompt, max_token, model, B, id_to_words_ref, tau)

In [ ]:
generate_response(X, id_to_words_ref)


**Choosing a Decoding Strategy**

When the objective is to generate predictable, high-probability sequences, greedy decoding or beam search may be appropriate. Beam search explores multiple candidate sequences and selects the one with the highest cumulative score among the candidates it retains.

When the objective is to generate more diverse or creative responses, sampling-based methods such as Top-K and Top-P are generally more suitable because they introduce randomness into token selection.

However, **high sequence probability does not necessarily mean high factual accuracy**. The decoding strategy controls how tokens are selected, while factual correctness also depends on the model's knowledge, training, context, and grounding.

In production conversational LLMs, temperature and Top-P sampling are commonly used to balance output quality and diversity.

# Text Generation and Decoding in Production LLMs

## 1. Introduction

In our TinyGPT implementation, we explored five decoding strategies:

1. **Greedy Decoding** — Select the token with the highest probability.
2. **Temperature Sampling** — Adjust the probability distribution and randomly sample the next token.
3. **Top-K Sampling** — Sample from the K most probable tokens.
4. **Top-P (Nucleus) Sampling** — Sample from the smallest set of tokens whose cumulative probability reaches a threshold.
5. **Beam Search** — Maintain multiple candidate sequences and select the best sequences using cumulative scores.

These methods demonstrate the fundamental process of autoregressive text generation.

In production, LLM systems use these same underlying principles, but additional techniques are required to improve efficiency, control output, and handle real-world applications.

## 2. How Decoding Is Used in Production

Different applications may require different decoding strategies.

| Decoding strategy | Typical use case | Considerations |
|---|---|---|
| Greedy | Deterministic generation, simple baselines | Fast, but may produce repetitive or suboptimal outputs |
| Temperature | Control randomness in sampling | Usually combined with other sampling controls |
| Top-K | Restrict sampling to likely tokens | Fixed candidate count may be too restrictive or too broad |
| Top-P | General-purpose conversational and creative generation | Adapts candidate count to the probability distribution |
| Beam Search | Machine translation, speech recognition, sequence generation | Explores multiple sequences but costs more computation |

**Important:** Modern conversational LLMs commonly use sampling-based decoding, often with temperature and Top-P controls. Beam search is more common in applications where finding a high-scoring sequence is particularly important.

No single decoding strategy is optimal for every application.

## 3. Production Techniques Beyond Basic Decoding

### 3.1 Log Probabilities

In our TinyGPT beam search, we calculate cumulative scores by multiplying probabilities:

`new_score = previous_score × next_token_probability`

When generating many tokens, repeated multiplication can produce extremely small numbers, eventually causing numerical underflow.

Production beam search typically uses log probabilities instead:

`new_score = previous_log_score + log(next_token_probability)`

In PyTorch, `torch.log_softmax()` can calculate log probabilities directly.

This is mathematically equivalent for ranking fixed-length sequences, but more numerically stable.

### 3.2 Stopping Conditions and Special Tokens

Our TinyGPT generates a fixed number of tokens using `max_token`.

Production LLMs generally support additional stopping conditions:

- **EOS (End-of-Sequence)** — Indicates that generation should stop.
- **Maximum output tokens** — Prevents generation from continuing indefinitely.
- **Stop sequences** — Stop generation when a specified token sequence appears.
- **Application-specific stopping rules** — For example, completion of a structured response.

An important distinction is that production systems may finish generating before reaching the maximum token limit.

### 3.3 KV Cache (Key-Value Cache)

In our TinyGPT implementation, we pass the entire generated sequence back into the Transformer at every step.

This means previously processed tokens are repeatedly used to recompute attention information.

Production decoder-only Transformers commonly use a **KV cache**, which stores the key and value tensors from previous tokens.

During subsequent generation steps, the model reuses cached keys and values and computes new attention information primarily for the new token.

This significantly reduces redundant computation during autoregressive generation, although the cache consumes memory.

### 3.4 Prefill and Decode Phases

Production LLM inference is often divided into two phases.

**Prefill:** The model processes the input prompt and prepares the attention cache.

**Decode:** The model generates new tokens autoregressively, usually one token per sequence per decoding iteration.

These phases have different computational characteristics and are important when optimizing inference performance.

### 3.5 Repetition and Length Controls

Some decoding systems provide additional controls, such as:

- **Repetition penalty** — Discourages repeated tokens.
- **Frequency penalty** — Adjusts token scores based on how frequently tokens have appeared.
- **Presence penalty** — Adjusts token scores based on whether tokens have already appeared.
- **Length normalization** — Helps beam search compare sequences of different lengths.

These controls are optional and their behavior depends on the model and serving framework.

### 3.6 Constrained and Structured Decoding

Production applications may require output in a specific format, such as JSON.

Instead of allowing the model to select any vocabulary token, constrained decoding can restrict which tokens are valid at each generation step.

For example, when generating JSON, the decoder can enforce a grammar or schema so that invalid token sequences are excluded.

This is particularly useful for applications involving APIs, tools, and structured data extraction.

### 3.7 Inference Performance Optimization

Production systems must serve many users efficiently.

Common techniques include:

- **Continuous batching** — Dynamically group active generation requests to improve GPU utilization.
- **Paged KV cache management** — Manage attention-cache memory efficiently.
- **Quantization** — Reduce model weight and sometimes cache precision to lower memory and computational requirements.
- **Speculative decoding** — Use a smaller or faster model to propose tokens that a larger model verifies, potentially accelerating generation without changing the target model's output distribution.

These techniques primarily improve inference efficiency rather than change the fundamental autoregressive generation process.

## 4. Recommended Topics for Further Learning

After completing TinyGPT, learners can explore the following topics in order.

| Priority | Topic | Learning objective |
|---|---|---|
| 1 | Log probabilities | Understand numerical stability in sequence scoring |
| 2 | EOS and stopping conditions | Understand how generation terminates |
| 3 | KV caching | Understand how Transformers avoid redundant attention computation |
| 4 | Prefill vs. decode | Understand the two main phases of LLM inference |
| 5 | Token penalties and length normalization | Understand practical generation controls |
| 6 | Structured decoding | Understand how LLMs produce constrained outputs |
| 7 | Batching and inference serving | Understand how multiple requests are handled efficiently |
| 8 | Speculative decoding and quantization | Explore advanced inference optimization |

Learners interested in the underlying algorithms can also investigate **contrastive search, diverse beam search, and typical sampling**, although these are not prerequisites for understanding production inference.

## 5. Suggested Practical Exercises

For learners who want to extend TinyGPT:

**Exercise 1: Log-Probability Beam Search**

Replace probability multiplication with log-probability addition. Compare the selected sequences with the original implementation.

**Exercise 2: EOS Token**

Introduce an EOS token into the vocabulary and modify generation to stop when it is selected. This may require modifying the training data and retraining the model.

**Exercise 3: KV Cache**

Study how attention keys and values can be reused between generation steps. This is a more advanced exercise because it requires changes to the attention implementation.

**Exercise 4: Compare Decoding Strategies**

Using the same prompt, compare greedy decoding, Top-K, Top-P, and beam search under different temperature settings. Record output diversity, cumulative scores, and generation time.

## 6. Conclusion

The objective of TinyGPT is to understand the **fundamental mechanisms of token generation**, not to reproduce the complexity of a production LLM inference engine.

By implementing greedy decoding, temperature sampling, Top-K, Top-P, and beam search, learners have covered the essential concepts of selecting the next token and constructing an output sequence.

Production systems build upon these foundations with additional mechanisms for stopping, numerical stability, memory efficiency, output constraints, and high-throughput inference.

**The recommended next conceptual topic is KV caching**, after a brief introduction to log probabilities and stopping conditions, because it connects directly to the attention mechanism already studied in TinyGPT.

## END